# Fink/LSST — Detection Threshold vs. Focal-Plane Radius

This notebook reads the src Parquet files saved by `01_fink_block_lightcurves.ipynb`
and produces:

1. **All-band focal-plane heatmaps** (1×2 dual panel): median psfFlux and median magnitude
   aggregated over all bands together (all-band combined).

2. **Per-band focal-plane heatmaps**: each photometric band `u g r i z y` is treated
   **independently** — fluxes from different bands are **never mixed** in the same heatmap.
   Each band gets its own colour scale derived from its own flux/magnitude distribution.

3. **Error plots** (value ± uncertainty vs. radial distance $r$):
   - $\langle F_{\rm psf} \rangle$ vs. $r$  [nJy vs. mm]
   - $\langle m \rangle$ vs. $r$  [mag vs. mm]
   Both all-band and per-band versions.

### Motivation

The median PSF flux of *detected* src alerts is a proxy for the **per-visit detection
threshold**: a higher median flux means fainter sources are not detected, i.e. the
threshold is higher (less sensitive).  If the PSF degrades or the background rises at
large focal-plane radii, we expect the median flux threshold to **increase** (and the
median magnitude to **decrease**) with $r = \sqrt{x^2 + y^2}$.

No functional fit is applied here — the goal is to **visualise the shape** of these
radial profiles before choosing a fitting function.

### Per-CCD uncertainty

The uncertainty on the median is estimated via the **MAD-based standard error**:

$$\sigma_{\rm med} \approx 1.4826 \times \frac{\text{MAD}}{\sqrt{N}}$$

where MAD $= \text{median}(|x_i - \text{median}(x)|)$ and $N$ is the number of valid
src alerts on the CCD.

---

- **Author:** Sylvie Dagoret-Campagne
- **Affiliation:** IJCLab / IN2P3 / CNRS
- **Creation date:** 2026-04-03
- **Last update:** 2026-04-03
- **Subject:** Fink alert broker — Rubin LSST detection threshold vs. focal-plane radius

## 1. Imports & configuration

In [ ]:
import os
import glob
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.colors as mcolors

warnings.filterwarnings("ignore")

print(f"pandas  version : {pd.__version__}")
print(f"numpy   version : {np.__version__}")

In [ ]:
# Enable interactive matplotlib backend when ipympl is available
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → falling back to %matplotlib inline")
    print("Install with:  pip install ipympl")

In [ ]:
# ── Notebook tag & paths ──────────────────────────────────────────────────────
NB_TAG = "FINK_BLOCK_LC_01"
DIR_DATA = f"data_{NB_TAG}"  # input: written by notebook 01
DIR_FIGS = "figs_FINK_BLOCK_LC_05c"  # output figures for this notebook
os.makedirs(DIR_FIGS, exist_ok=True)

# ── Rubin LSSTCam focal-plane geometry CSV ────────────────────────────────────
PATH_GEOM_CSV = "data_FocalPlane/ccd_geometry.csv"

# ── Groups to use (photometrically stable Gaia stars) ─────────────────────────
GROUPS_OF_INTEREST = ["gaia_star_stable", "gaia_star_stable_hq"]

# ── Column names in the src parquet files ─────────────────────────────────────
COL_DETECTOR = "r:detector"
COL_PSF = "r:psfFlux"  # [nJy]
COL_PSFERR = "r:psfFluxErr"  # [nJy]
COL_MJD = "r:midpointMjdTai"
COL_BAND = "r:band"

# ── AB zero-point: m = -2.5*log10(f_nJy) + 31.4 ─────────────────────────────
AB_ZP = 31.4  # mag  (AB zero-point for nJy)

# ── MAD → sigma scale factor (for a Gaussian distribution) ───────────────────
MAD_SCALE = 1.4826

# ── Band colours (consistent with notebooks 01–05) ────────────────────────────
BAND_COLORS = {
    "u": "#9b59b6",
    "g": "#2ecc71",
    "r": "#e74c3c",
    "i": "#e67e22",
    "z": "#3498db",
    "y": "#795548",
}
BANDS = list("ugrizy")

# ── Matplotlib global settings ────────────────────────────────────────────────
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 9,
    }
)


def savefig(name):
    """Save current figure as both PDF and PNG in DIR_FIGS."""
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS, f"{name}.{ext}"), bbox_inches="tight")
    print(f"  → saved {name}.{{pdf,png}}")


print(f"Data directory   : {os.path.abspath(DIR_DATA)}")
print(f"Geometry CSV     : {os.path.abspath(PATH_GEOM_CSV)}")
print(f"Figure directory : {os.path.abspath(DIR_FIGS)}")
print(f"AB zero-point    : {AB_ZP} mag  (m = -2.5*log10(f_nJy) + {AB_ZP})")

## 2. Utility functions

In [ ]:
def flux_nJy_to_mag(flux_nJy, zero_point=AB_ZP):
    """
    Convert PSF flux in nJy to AB magnitude.

    m = -2.5 * log10(f_nJy) + zero_point

    Non-positive flux values are returned as NaN.

    Parameters
    ----------
    flux_nJy   : array-like  — PSF flux [nJy]
    zero_point : float       — AB zero-point offset (default 31.4 for nJy)

    Returns
    -------
    mag : ndarray  — AB magnitude (NaN where flux <= 0)
    """
    f = np.asarray(flux_nJy, dtype=float)
    with np.errstate(invalid="ignore", divide="ignore"):
        mag = np.where(f > 0, -2.5 * np.log10(f) + zero_point, np.nan)
    return mag


def mad_std(x):
    """
    Robust standard deviation estimate via the Median Absolute Deviation.

    sigma_MAD = 1.4826 * median(|x - median(x)|)

    Parameters
    ----------
    x : array-like — 1-D data (NaN values are ignored)

    Returns
    -------
    float — MAD-based sigma estimate (NaN if fewer than 2 finite values)
    """
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) < 2:
        return np.nan
    return MAD_SCALE * float(np.median(np.abs(x - np.median(x))))


def median_error(x):
    """
    Standard error on the median via the MAD:

    sigma_med ≈ 1.4826 * MAD / sqrt(N)

    Parameters
    ----------
    x : array-like — 1-D data (NaN values are ignored)

    Returns
    -------
    float — uncertainty on the median
    """
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    n = len(x)
    if n < 2:
        return np.nan
    return mad_std(x) / np.sqrt(n)


print("Utility functions defined: flux_nJy_to_mag, mad_std, median_error.")

## 3. Load src Parquet files and concatenate

In [ ]:
# Discover all src parquet files on disk
src_files = sorted(glob.glob(os.path.join(DIR_DATA, "*_src.parquet")))


def group_from_path(path):
    return os.path.basename(path).replace("_src.parquet", "")


groups_src = {group_from_path(p): p for p in src_files}

print(f"src parquet files found on disk: {len(groups_src)}")
for g in groups_src:
    print(f"  {g}")

In [ ]:
# Load and concatenate only the groups of interest
dfs_to_concat = []

for group in GROUPS_OF_INTEREST:
    if group not in groups_src:
        print(f"  WARNING: group '{group}' not found on disk — skipped.")
        continue
    df_g = pd.read_parquet(groups_src[group])
    df_g["_group"] = group
    dfs_to_concat.append(df_g)
    print(f"  Loaded {group}: {len(df_g):,} rows")

if len(dfs_to_concat) == 0:
    raise RuntimeError(
        f"None of the requested groups {GROUPS_OF_INTEREST} were found in {DIR_DATA}. Run notebook 01 first."
    )

df_src = pd.concat(dfs_to_concat, ignore_index=True)
print(f"\nTotal src rows loaded : {len(df_src):,}")

## 4. Schema inspection & column validation

In [ ]:
print("Columns in src table:")
print(df_src.dtypes.to_string())

In [ ]:
# Validate required columns
required = [COL_DETECTOR, COL_PSF, COL_PSFERR]
missing = [c for c in required if c not in df_src.columns]
if missing:
    raise KeyError(f"Missing required columns: {missing}")
print("All required columns present.")
print(f"Unique detectors (CCDs) : {df_src[COL_DETECTOR].nunique()}")
print(f"Detector id range       : {df_src[COL_DETECTOR].min()} – {df_src[COL_DETECTOR].max()}")

## 5. Compute magnitude column

$$m = -2.5 \log_{10}(F_{\rm psf}\,[{\rm nJy}]) + 31.4$$

Only positive-flux detections are kept; non-detections and negative DIA residuals
are masked to NaN.

In [ ]:
df_src = df_src.copy()

# Keep only strictly positive flux values for the threshold proxy
valid_flux = (df_src[COL_PSF].values > 0) & np.isfinite(df_src[COL_PSF].values)

# psfFlux: set non-valid to NaN
flux_vals = df_src[COL_PSF].values.astype(float)
flux_vals[~valid_flux] = np.nan
df_src["psfFlux"] = flux_vals

# Magnitude from psfFlux
df_src["mag"] = flux_nJy_to_mag(df_src["psfFlux"].values)

n_valid = (~np.isnan(df_src["psfFlux"])).sum()
n_invalid = np.isnan(df_src["psfFlux"]).sum()
print(f"psfFlux valid   : {n_valid:,}")
print(f"psfFlux masked  : {n_invalid:,}  (non-positive → NaN)")
print(f"psfFlux range   : [{np.nanmin(df_src['psfFlux']):.2f},  {np.nanmax(df_src['psfFlux']):.2f}] nJy")
print(f"mag range       : [{np.nanmin(df_src['mag']):.2f},  {np.nanmax(df_src['mag']):.2f}] mag")

## 6. All-band per-CCD aggregation: median psfFlux and median magnitude

**All bands are aggregated together** in this section only, to give a global
view of the detection threshold over the full focal plane.

For each CCD:
- **n_src** : number of valid src detections (all bands)
- **flux_med** : median psfFlux [nJy]
- **flux_err** : MAD-based standard error on the median flux
- **mag_med** : median magnitude [mag]
- **mag_err** : MAD-based standard error on the median magnitude

> **Note:** Mixing bands here is acceptable for a global focal-plane overview,
> but the physical interpretation is limited since each band probes a different
> sky brightness regime.  Per-band heatmaps (Sections 9–10) are the scientifically
> meaningful product.

In [ ]:
df_valid = df_src.dropna(subset=["psfFlux", "mag"])

ccd_stats = (
    df_valid.groupby(COL_DETECTOR)
    .agg(
        n_src=("psfFlux", "count"),
        flux_med=("psfFlux", "median"),
        flux_std=("psfFlux", "std"),
        flux_mad=("psfFlux", lambda x: mad_std(x)),
        flux_err=("psfFlux", lambda x: median_error(x)),
        mag_med=("mag", "median"),
        mag_std=("mag", "std"),
        mag_mad=("mag", lambda x: mad_std(x)),
        mag_err=("mag", lambda x: median_error(x)),
    )
    .reset_index()
    .rename(columns={COL_DETECTOR: "detector"})
)

print(f"CCDs with at least one valid src (all bands) : {len(ccd_stats)}")
print(ccd_stats[["detector", "n_src", "flux_med", "flux_err", "mag_med", "mag_err"]].describe())

In [ ]:
print("Per-CCD statistics, all bands (first 20 rows):")
display(ccd_stats[["detector", "n_src", "flux_med", "flux_err", "mag_med", "mag_err"]].head(20))

## 7. Load focal-plane geometry and merge

In [ ]:
if not os.path.exists(PATH_GEOM_CSV):
    raise FileNotFoundError(f"Focal-plane geometry CSV not found: {PATH_GEOM_CSV}")

geom = pd.read_csv(PATH_GEOM_CSV)
print(f"Geometry CSV loaded : {len(geom)} CCDs")
print(geom.columns.tolist())

# Compute radial distance of each CCD centre from the camera optical axis
geom["r_mm"] = np.sqrt(geom["x_center"] ** 2 + geom["y_center"] ** 2)

# Merge geometry with all-band per-CCD statistics
geom_stats = geom.merge(
    ccd_stats[["detector", "n_src", "flux_med", "flux_err", "mag_med", "mag_err"]],
    on="detector",
    how="left",
)

n_no_data = geom_stats["flux_med"].isna().sum()
print(f"CCDs with data    : {geom_stats['flux_med'].notna().sum()}")
print(f"CCDs without data : {n_no_data}  (will appear grey in heatmaps)")

## 8. Focal-plane heatmap helper

In [ ]:
def draw_focal_plane(
    ax,
    geom_df,
    value_col,
    cmap,
    norm,
    title,
    colorbar_label,
    fig,
    show_ccd_label=True,
):
    """
    Draw a single focal-plane heatmap panel on axes *ax*.

    Parameters
    ----------
    ax             : matplotlib Axes
    geom_df        : pd.DataFrame  — geometry merged with statistics
    value_col      : str           — column to colour-code
    cmap           : colormap name or object
    norm           : matplotlib Normalize instance
    title          : str           — subplot title
    colorbar_label : str           — colour-bar label
    fig            : matplotlib Figure
    show_ccd_label : bool          — print detector id + name inside each patch
    """
    xmin = geom_df[[f"corner{i}_x" for i in range(4)]].min().min()
    xmax = geom_df[[f"corner{i}_x" for i in range(4)]].max().max()
    ymin = geom_df[[f"corner{i}_y" for i in range(4)]].min().min()
    ymax = geom_df[[f"corner{i}_y" for i in range(4)]].max().max()

    cmap_obj = plt.get_cmap(cmap) if isinstance(cmap, str) else cmap

    for _, row in geom_df.iterrows():
        corners = [(row[f"corner{j}_x"], row[f"corner{j}_y"]) for j in range(4)]
        val = row[value_col]
        face_color = "lightgrey" if np.isnan(val) else cmap_obj(norm(val))

        poly = patches.Polygon(
            corners,
            facecolor=face_color,
            edgecolor="black",
            linewidth=0.2,
        )
        ax.add_patch(poly)

        if show_ccd_label:
            ax.text(
                row["x_center"],
                row["y_center"],
                f"{int(row['detector'])}\n{row['name']}",
                ha="center",
                va="center",
                fontsize=5.5,
                fontweight="bold",
                color="k",
            )

    ax.set_aspect("equal")
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    ax.set_xlabel("Focal plane X  [mm]", fontsize=9)
    ax.set_ylabel("Focal plane Y  [mm]", fontsize=9)
    ax.set_title(title, fontsize=10)

    sm = plt.cm.ScalarMappable(cmap=cmap_obj, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label(colorbar_label, fontsize=8)
    if "mag" in title:
        cbar.ax.invert_yaxis()


def build_band_geom_stats(df_src_full, band, geom_df):
    """
    Compute per-CCD median flux and magnitude for a single photometric band,
    then merge with the focal-plane geometry.

    Each band is processed in complete isolation — no mixing with other bands.

    Parameters
    ----------
    df_src_full : pd.DataFrame  — full src table with 'psfFlux', 'mag', COL_BAND, COL_DETECTOR
    band        : str           — single band letter (e.g. 'r')
    geom_df     : pd.DataFrame  — focal-plane geometry (including 'r_mm')

    Returns
    -------
    geom_band : pd.DataFrame  — geometry + per-CCD statistics (NaN where no data)
    n_src     : int           — total number of valid src alerts in this band
    """
    df_b = df_src_full[df_src_full[COL_BAND] == band].dropna(subset=["psfFlux", "mag"])

    if len(df_b) == 0:
        geom_band = geom_df.copy()
        for col in ("n_src", "flux_med", "flux_err", "mag_med", "mag_err"):
            geom_band[col] = np.nan
        return geom_band, 0

    band_stats = (
        df_b.groupby(COL_DETECTOR)
        .agg(
            n_src=("psfFlux", "count"),
            flux_med=("psfFlux", "median"),
            flux_err=("psfFlux", lambda x: median_error(x)),
            mag_med=("mag", "median"),
            mag_err=("mag", lambda x: median_error(x)),
        )
        .reset_index()
        .rename(columns={COL_DETECTOR: "detector"})
    )

    geom_band = geom_df.merge(band_stats, on="detector", how="left")
    return geom_band, int(len(df_b))


print("draw_focal_plane() and build_band_geom_stats() defined.")

## 9. All-band heatmap: median psfFlux and median magnitude on the focal plane

**Warning:** this panel mixes all photometric bands.  It provides a global
overview of which CCDs have data, but the flux values are not physically
comparable across bands.  See Sections 10–11 for per-band heatmaps.

- **Left panel** : median psfFlux per CCD — sequential colour scale (plasma).
- **Right panel** : median magnitude per CCD — inverted sequential colour scale.

In [ ]:
# ── Colour-scale limits for the all-band combined heatmap ─────────────────────
flux_vals_ccd = geom_stats["flux_med"].dropna()
mag_vals_ccd = geom_stats["mag_med"].dropna()

vmin_flux = float(np.percentile(flux_vals_ccd, 5)) if len(flux_vals_ccd) > 0 else 0
vmax_flux = float(np.percentile(flux_vals_ccd, 95)) if len(flux_vals_ccd) > 0 else 1
norm_flux = mcolors.Normalize(vmin=vmin_flux, vmax=vmax_flux)
cmap_flux = "plasma"

vmin_mag = float(np.percentile(mag_vals_ccd, 5)) if len(mag_vals_ccd) > 0 else 15
vmax_mag = float(np.percentile(mag_vals_ccd, 95)) if len(mag_vals_ccd) > 0 else 24
norm_mag = mcolors.Normalize(vmin=vmin_mag, vmax=vmax_mag)
cmap_mag = "plasma_r"

print(f"All-band flux colour range : [{vmin_flux:.2f},  {vmax_flux:.2f}] nJy")
print(f"All-band mag  colour range : [{vmin_mag:.2f},  {vmax_mag:.2f}] mag")

In [ ]:
fig, (ax_flux, ax_mag) = plt.subplots(
    1,
    2,
    figsize=(18, 9),
    constrained_layout=True,
)

draw_focal_plane(
    ax=ax_flux,
    geom_df=geom_stats,
    value_col="flux_med",
    cmap=cmap_flux,
    norm=norm_flux,
    title=r"All bands — Median $F_{\rm psf}$ per CCD  [nJy]",
    colorbar_label=r"median $F_{\rm psf}$  [nJy]  (all bands)",
    fig=fig,
)

draw_focal_plane(
    ax=ax_mag,
    geom_df=geom_stats,
    value_col="mag_med",
    cmap=cmap_mag,
    norm=norm_mag,
    title=r"All bands — Median magnitude $m$ per CCD  [mag]",
    colorbar_label=r"median $m = -2.5\log_{10}(F_{\rm psf}) + 31.4$  [mag]  (all bands)",
    fig=fig,
)

fig.suptitle(
    r"LSSTCam Focal Plane — Detection threshold proxy  "
    r"(all bands combined — for global overview only)  "
    f"groups: {', '.join(GROUPS_OF_INTEREST)}",
    fontsize=11,
)

savefig("focal_plane_FluxThreshold_allbands_combined")
plt.show()

## 10. Per-band focal-plane heatmaps: individual figures per band

Each band is processed **completely independently** — there is no mixing
of fluxes from different bands.  For each of the six bands `u g r i z y`:

- A dedicated 1×2 figure is produced.
- **Left panel**: median psfFlux per CCD, with a colour scale derived
  exclusively from the 5th–95th percentile of that band's flux distribution.
- **Right panel**: median magnitude per CCD, with its own independent colour scale.
- CCDs with no detection in the band appear in light grey.

This is the scientifically meaningful product for studying the radial
variation of the detection threshold per photometric band.

In [ ]:
if COL_BAND not in df_src.columns:
    print(f"Column '{COL_BAND}' not found — skipping per-band focal-plane heatmaps.")
else:
    for band in BANDS:
        geom_b, n_src_b = build_band_geom_stats(df_src, band, geom)

        if n_src_b == 0:
            print(f"  band {band}: no valid src — skipped.")
            continue

        # ── Colour scale derived exclusively from this band ───────────────────
        flux_b = geom_b["flux_med"].dropna()
        mag_b = geom_b["mag_med"].dropna()
        n_ccd_b = int(flux_b.notna().sum()) if hasattr(flux_b, "notna") else len(flux_b)

        vmin_f_b = float(np.percentile(flux_b, 5)) if len(flux_b) > 0 else 0
        vmax_f_b = float(np.percentile(flux_b, 95)) if len(flux_b) > 0 else 1
        vmin_m_b = float(np.percentile(mag_b, 5)) if len(mag_b) > 0 else 15
        vmax_m_b = float(np.percentile(mag_b, 95)) if len(mag_b) > 0 else 24

        # Guard against degenerate ranges
        if vmax_f_b <= vmin_f_b:
            vmax_f_b = vmin_f_b + 1.0
        if vmax_m_b <= vmin_m_b:
            vmax_m_b = vmin_m_b + 0.5

        norm_f_b = mcolors.Normalize(vmin=vmin_f_b, vmax=vmax_f_b)
        norm_m_b = mcolors.Normalize(vmin=vmin_m_b, vmax=vmax_m_b)

        print(
            f"  band {band}: {n_src_b:,} src  | "
            f"flux [{vmin_f_b:.1f}, {vmax_f_b:.1f}] nJy  | "
            f"mag [{vmin_m_b:.2f}, {vmax_m_b:.2f}]"
        )

        fig_b, (ax_f, ax_m) = plt.subplots(
            1,
            2,
            figsize=(18, 9),
            constrained_layout=True,
        )

        # Left: median psfFlux heatmap — this band only
        draw_focal_plane(
            ax=ax_f,
            geom_df=geom_b,
            value_col="flux_med",
            cmap="plasma",
            norm=norm_f_b,
            title=rf"Band {band} — Median $F_{{\rm psf}}$ per CCD  [nJy]",
            colorbar_label=rf"band {band} — median $F_{{\rm psf}}$  [nJy]",
            fig=fig_b,
        )

        # Right: median magnitude heatmap — this band only
        draw_focal_plane(
            ax=ax_m,
            geom_df=geom_b,
            value_col="mag_med",
            cmap="plasma_r",
            norm=norm_m_b,
            title=rf"Band {band} — Median $m$ per CCD  [mag]",
            colorbar_label=rf"band {band} — median $m$  [mag]",
            fig=fig_b,
        )

        fig_b.suptitle(
            rf"LSSTCam Focal Plane — Band {band} — Detection threshold  "
            rf"(N_src={n_src_b:,}  |  colour scale: band {band} only)",
            fontsize=11,
        )

        savefig(f"focal_plane_FluxThreshold_band{band}")
        plt.show()

## 11. Multi-band summary heatmap: all bands on a single grid figure

A compact N_active_bands × 2 grid showing all bands simultaneously.
- Row = one band
- Left column = median psfFlux
- Right column = median magnitude

**Each row has its own independent colour scale** so that intra-band spatial
patterns are visible regardless of the large inter-band flux differences.
The colour scale of row `b` is derived solely from band `b` data.

In [ ]:
if COL_BAND not in df_src.columns:
    print(f"Column '{COL_BAND}' not found — skipping multi-band summary heatmap.")
else:
    # Pre-compute per-band geometry stats (band isolation guaranteed)
    band_geom_data = {}
    for band in BANDS:
        geom_b, n_src_b = build_band_geom_stats(df_src, band, geom)
        band_geom_data[band] = (geom_b, n_src_b)

    active_bands = [b for b in BANDS if band_geom_data[b][1] > 0]
    n_active = len(active_bands)
    print(f"Active bands with data: {active_bands}")

    if n_active == 0:
        print("No band has valid src data — skipping multi-band heatmap.")
    else:
        fig_all, axes_all = plt.subplots(
            n_active,
            2,
            figsize=(18, 9 * n_active),
            constrained_layout=True,
        )

        # Ensure axes_all is always 2-D
        if n_active == 1:
            axes_all = axes_all[np.newaxis, :]

        for row_idx, band in enumerate(active_bands):
            geom_b, n_src_b = band_geom_data[band]

            # Colour scale from this band only
            flux_b = geom_b["flux_med"].dropna()
            mag_b = geom_b["mag_med"].dropna()

            vmin_f_b = float(np.percentile(flux_b, 5)) if len(flux_b) > 0 else 0
            vmax_f_b = float(np.percentile(flux_b, 95)) if len(flux_b) > 0 else 1
            vmin_m_b = float(np.percentile(mag_b, 5)) if len(mag_b) > 0 else 15
            vmax_m_b = float(np.percentile(mag_b, 95)) if len(mag_b) > 0 else 24

            if vmax_f_b <= vmin_f_b:
                vmax_f_b = vmin_f_b + 1.0
            if vmax_m_b <= vmin_m_b:
                vmax_m_b = vmin_m_b + 0.5

            norm_f_b = mcolors.Normalize(vmin=vmin_f_b, vmax=vmax_f_b)
            norm_m_b = mcolors.Normalize(vmin=vmin_m_b, vmax=vmax_m_b)

            # Left column: flux
            draw_focal_plane(
                ax=axes_all[row_idx, 0],
                geom_df=geom_b,
                value_col="flux_med",
                cmap="plasma",
                norm=norm_f_b,
                title=rf"Band {band} — Median $F_{{\rm psf}}$  [nJy]  (N={n_src_b:,})",
                colorbar_label=rf"band {band} — $F_{{\rm psf}}$  [nJy]",
                fig=fig_all,
            )

            # Right column: magnitude
            draw_focal_plane(
                ax=axes_all[row_idx, 1],
                geom_df=geom_b,
                value_col="mag_med",
                cmap="plasma_r",
                norm=norm_m_b,
                title=rf"Band {band} — Median $m$  [mag]  (N={n_src_b:,})",
                colorbar_label=rf"band {band} — $m$  [mag]",
                fig=fig_all,
            )

        fig_all.suptitle(
            r"LSSTCam Focal Plane — Detection threshold per band  "
            r"(left: median $F_{\rm psf}$ / right: median $m$  —  "
            r"each row: independent band-only colour scale)",
            fontsize=12,
        )

        savefig("focal_plane_FluxThreshold_all_bands_grid")
        plt.show()

## 12. Error plot: median psfFlux vs. focal-plane radius $r$ (all bands)

Each point is one CCD.  The error bars represent $\sigma_{\rm med} \approx 1.4826 \times {\rm MAD}/\sqrt{N}$.

**Expected signal:** if the PSF broadens or the background rises at large radii,
fainter sources are not detected → the median detected flux should **increase** with $r$.

In [ ]:
# Select only CCDs with valid data (all-band stats)
geom_plot = geom_stats.dropna(subset=["flux_med", "mag_med"]).copy()
print(f"CCDs used in radial plots (all bands): {len(geom_plot)}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(
    1,
    2,
    figsize=(16, 5.5),
    constrained_layout=True,
)

# ── Left: flux vs r ───────────────────────────────────────────────────────────
sc1 = ax1.scatter(
    geom_plot["r_mm"],
    geom_plot["flux_med"],
    c=geom_plot["n_src"],
    cmap="viridis",
    s=40,
    zorder=4,
)
ax1.errorbar(
    geom_plot["r_mm"],
    geom_plot["flux_med"],
    yerr=geom_plot["flux_err"],
    fmt="none",
    ecolor="grey",
    elinewidth=0.8,
    capsize=2,
    alpha=0.6,
    zorder=3,
)
fig.colorbar(sc1, ax=ax1, fraction=0.03, pad=0.01).set_label("N src per CCD (all bands)", fontsize=8)
ax1.set_xlabel(r"$r = \sqrt{x^2+y^2}$  [mm]", fontsize=10)
ax1.set_ylabel(r"Median $F_{\rm psf}$  [nJy]", fontsize=10)
ax1.set_title(r"All bands — Median $F_{\rm psf}$ vs. focal-plane radius", fontsize=10)
ax1.set_xlim(-1, geom_plot["r_mm"].max() * 1.05)
ax1.set_ylim(bottom=0)

# ── Right: magnitude vs r ─────────────────────────────────────────────────────
sc2 = ax2.scatter(
    geom_plot["r_mm"],
    geom_plot["mag_med"],
    c=geom_plot["n_src"],
    cmap="viridis",
    s=40,
    zorder=4,
)
ax2.errorbar(
    geom_plot["r_mm"],
    geom_plot["mag_med"],
    yerr=geom_plot["mag_err"],
    fmt="none",
    ecolor="grey",
    elinewidth=0.8,
    capsize=2,
    alpha=0.6,
    zorder=3,
)
fig.colorbar(sc2, ax=ax2, fraction=0.03, pad=0.01).set_label("N src per CCD (all bands)", fontsize=8)
ax2.invert_yaxis()
ax2.set_xlabel(r"$r = \sqrt{x^2+y^2}$  [mm]", fontsize=10)
ax2.set_ylabel(r"Median $m$  [mag]", fontsize=10)
ax2.set_title(r"All bands — Median $m$ vs. focal-plane radius", fontsize=10)
ax2.set_xlim(-1, geom_plot["r_mm"].max() * 1.05)

fig.suptitle(
    r"Detection threshold proxy — all bands combined  "
    f"(groups: {', '.join(GROUPS_OF_INTEREST)},  N_CCD={len(geom_plot)})",
    fontsize=11,
)

savefig("FluxThreshold_flux_and_mag_vs_radius_allbands")
plt.show()

## 13. Per-band error plots: median psfFlux and median magnitude vs. $r$

The radial analysis is repeated for each photometric band `u g r i z y` separately.
Each band is plotted with its own colour to allow easy visual comparison.  This
reveals whether the radial variation of the detection threshold is chromatic.

In [ ]:
if COL_BAND not in df_src.columns:
    print(f"Column '{COL_BAND}' not found — skipping per-band radial plots.")
else:
    for band in BANDS:
        df_band = df_src[df_src[COL_BAND] == band].dropna(subset=["psfFlux", "mag"])

        if len(df_band) == 0:
            print(f"  band {band}: no valid src — skipped.")
            continue

        # Per-CCD statistics for this band (computed from band data only)
        band_stats = (
            df_band.groupby(COL_DETECTOR)
            .agg(
                n_src=("psfFlux", "count"),
                flux_med=("psfFlux", "median"),
                flux_err=("psfFlux", lambda x: median_error(x)),
                mag_med=("mag", "median"),
                mag_err=("mag", lambda x: median_error(x)),
            )
            .reset_index()
            .rename(columns={COL_DETECTOR: "detector"})
        )

        geom_band = geom[["detector", "name", "x_center", "y_center", "r_mm"]].merge(
            band_stats,
            on="detector",
            how="inner",
        )

        if len(geom_band) == 0:
            print(f"  band {band}: no CCD matched — skipped.")
            continue

        bcolor = BAND_COLORS.get(band, "grey")
        n_ccd_band = len(geom_band)
        n_src_band = len(df_band)

        fig_b, (ax_f, ax_m) = plt.subplots(
            1,
            2,
            figsize=(14, 5),
            constrained_layout=True,
        )

        ax_f.scatter(geom_band["r_mm"], geom_band["flux_med"], color=bcolor, s=35, zorder=4)
        ax_f.errorbar(
            geom_band["r_mm"],
            geom_band["flux_med"],
            yerr=geom_band["flux_err"],
            fmt="none",
            ecolor=bcolor,
            elinewidth=0.8,
            capsize=2,
            alpha=0.5,
            zorder=3,
        )
        ax_f.set_xlabel(r"$r$  [mm]", fontsize=10)
        ax_f.set_ylabel(r"Median $F_{\rm psf}$  [nJy]", fontsize=10)
        ax_f.set_title(rf"Band {band} — median $F_{{\rm psf}}$ vs. $r$", fontsize=10)
        ax_f.set_xlim(-1, geom_band["r_mm"].max() * 1.05)
        ax_f.set_ylim(bottom=0)

        ax_m.scatter(geom_band["r_mm"], geom_band["mag_med"], color=bcolor, s=35, zorder=4)
        ax_m.errorbar(
            geom_band["r_mm"],
            geom_band["mag_med"],
            yerr=geom_band["mag_err"],
            fmt="none",
            ecolor=bcolor,
            elinewidth=0.8,
            capsize=2,
            alpha=0.5,
            zorder=3,
        )
        ax_m.invert_yaxis()
        ax_m.set_xlabel(r"$r$  [mm]", fontsize=10)
        ax_m.set_ylabel(r"Median $m$  [mag]", fontsize=10)
        ax_m.set_title(rf"Band {band} — median $m$ vs. $r$", fontsize=10)
        ax_m.set_xlim(-1, geom_band["r_mm"].max() * 1.05)

        fig_b.suptitle(
            rf"Band {band} — Detection threshold vs. focal-plane radius "
            f"(N_CCD={n_ccd_band},  N_src={n_src_band:,})",
            fontsize=11,
        )

        savefig(f"FluxThreshold_flux_and_mag_vs_radius_band{band}")
        plt.show()

## 14. Summary table: per-CCD detection threshold statistics (all bands)

In [ ]:
summary_cols = [
    "detector",
    "name",
    "x_center",
    "y_center",
    "r_mm",
    "n_src",
    "flux_med",
    "flux_err",
    "mag_med",
    "mag_err",
]
df_summary = geom_plot[summary_cols].sort_values("r_mm").reset_index(drop=True)
print("Per-CCD detection threshold summary — all bands — sorted by radial distance:")
display(df_summary)

In [ ]:
# Save summary CSV alongside the figures
summary_path = os.path.join(DIR_FIGS, "ccd_flux_threshold_summary.csv")
df_summary.to_csv(summary_path, index=False)
print(f"Summary saved to: {summary_path}")